In [ ]:
import sys
import os

%pip install "numpy<2.0" transformer_lens scikit-learn matplotlib tqdm -q

if not os.path.exists('/content/MATS-take-home'):
    !git clone https://github.com/vedantgaur/MATS-take-home.git
else:
    %cd /content/MATS-take-home
    !git pull
    %cd /content

sys.path.append('/content/MATS-take-home')

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

from data.mess3_generator import Mess3Process, NonErgodicMess3Dataset
from models.train import get_toy_config, train_model
from transformer_lens import HookedTransformer
from analysis.geometry import extract_activations, calculate_cev, plot_cev, plot_2d_pca
from analysis.orthogonality import compare_all_processes

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
p1 = Mess3Process(alpha=0.60, x=0.15)
p2 = Mess3Process(alpha=0.79, x=0.11)
p3 = Mess3Process(alpha=0.60, x=0.50)

num_train_samples = 20000
seq_length = 16
dataset = NonErgodicMess3Dataset(
    num_samples=num_train_samples, 
    seq_length=seq_length, 
    processes=[p1, p2, p3]
)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

val_dataset = NonErgodicMess3Dataset(
    num_samples=1000, 
    seq_length=seq_length, 
    processes=[p1, p2, p3]
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
cfg = get_toy_config(vocab_size=3, d_model=64, n_ctx=seq_length)
model = HookedTransformer(cfg).to(device)

print("Starting training...")
loss_history = train_model(model, train_loader, epochs=150, lr=1e-3)

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.show()

In [ ]:
from analysis.geometry import extract_all_layers

print("Extracting activations across layers...")
acts_by_layer, labels = extract_all_layers(model, val_loader)

n_layers = len(acts_by_layer)

for layer_idx in range(n_layers):
    print(f"\n{'='*40}")
    print(f"ANALYZING LAYER {layer_idx}")
    print(f"{'='*40}")
    
    acts = acts_by_layer[layer_idx]
    
    pca_model, cev = calculate_cev(acts)
    dims_needed = plot_cev(cev, threshold=0.95)
    print(f"[Layer {layer_idx}] Dimensions required for 95% variance: {dims_needed}")
    
    plot_2d_pca(acts, labels, pca_model, pc_x=0, pc_y=1)
    
    plot_2d_pca(acts, labels, pca_model, pc_x=2, pc_y=3)
    
    print(f"[Layer {layer_idx}] Calculating pairwise subspace overlaps...")
    overlap_matrix = compare_all_processes(acts, labels, k_dims=2)
    
    plt.figure(figsize=(6, 5))
    plt.imshow(overlap_matrix, cmap='magma', vmin=0, vmax=1)
    plt.colorbar(label='Overlap Score (0=Orthogonal, 1=Parallel)')
    plt.xticks(ticks=[0, 1, 2], labels=['Process 1', 'Process 2', 'Process 3'])
    plt.yticks(ticks=[0, 1, 2], labels=['Process 1', 'Process 2', 'Process 3'])
    plt.title(f"Layer {layer_idx} Subspace Overlap")
    plt.show()
    
    print(f"[Layer {layer_idx}] Overlap Matrix:")
    print(np.round(overlap_matrix, 3))

In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_context_evolution(model, dataloader, pca_model):
    model.eval()
    
    last_layer = model.cfg.n_layers - 1
    hook_name = f"blocks.{last_layer}.hook_resid_post"
    
    batch_x, _, labels = next(iter(dataloader))
    batch_x = batch_x.to(model.cfg.device)
    
    with torch.no_grad():
        _, cache = model.run_with_cache(batch_x, names_filter=[hook_name])
        acts = cache[hook_name].cpu().numpy()
    
    positions_to_plot = [0, 3, 7, 14] 
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharex=True, sharey=True)
    
    for i, pos in enumerate(positions_to_plot):
        pos_acts = acts[:, pos, :]
        proj = pca_model.transform(pos_acts) 
        
        scatter = axes[i].scatter(proj[:, 0], proj[:, 1], c=labels.numpy(), cmap='viridis', alpha=0.7, s=15)
        axes[i].set_title(f"Position l={pos+1}")
        axes[i].set_xlabel("PC 1")
        if i == 0:
            axes[i].set_ylabel("PC 2")
            
    plt.suptitle("Activation Geometry Evolution Across Context Window", fontsize=16)
    plt.tight_layout()
    plt.show()

plot_context_evolution(model, val_loader, pca_model)

In [ ]:
import torch
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

all_acts = []
all_beliefs = []

model.eval()
last_layer = model.cfg.n_layers - 1
hook_name = f"blocks.{last_layer}.hook_resid_post"

with torch.no_grad():
    for batch_x, batch_beliefs, labels in val_loader:
        batch_x = batch_x.to(model.cfg.device)
        
        _, cache = model.run_with_cache(batch_x, names_filter=[hook_name])
        acts = cache[hook_name][:, -1, :].cpu().numpy() 
        
        if batch_beliefs.ndim == 3:
            beliefs = batch_beliefs[:, -1, :].cpu().numpy()
        else:
            beliefs = batch_beliefs.cpu().numpy()
            
        all_acts.append(acts)
        all_beliefs.append(beliefs)

acts = np.concatenate(all_acts, axis=0)
true_beliefs = np.concatenate(all_beliefs, axis=0)

X_train, X_test, y_train, y_test = train_test_split(acts, true_beliefs, test_size=0.2, random_state=42)

probe = Ridge(alpha=1.0)
probe.fit(X_train, y_train)

y_pred = probe.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"Linear Probe R^2 Score: {r2:.3f}")